# Transformer Architecture: Interactive Theory

> *"Attention is all you need — but understanding why requires linear algebra, probability, and careful analysis of computational complexity."*

This notebook provides interactive demonstrations of the mathematical foundations of the Transformer architecture. We implement attention from scratch, verify theoretical results numerically, visualise attention patterns, and explore efficient attention methods.

**Sections:**
1. Setup
2. Dot-Product Attention
3. Variance Scaling Proof
4. Softmax Properties
5. Attention as Kernel Smoothing
6. Multi-Head Attention
7. Positional Encodings
8. RoPE
9. Feed-Forward Networks
10. Normalisation Layers
11. Full Transformer Block
12. Parameter Counting
13. Computational Complexity
14. FlashAttention (Online Softmax)
15. LoRA
16. Attention Patterns & Interpretability

In [ ]:
import numpy as np
import numpy.linalg as la
from scipy import stats

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    mpl.rcParams.update({
        'figure.figsize': (10, 6),
        'figure.dpi': 120,
        'font.size': 13,
        'axes.titlesize': 15,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'legend.framealpha': 0.85,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'savefig.bbox': 'tight',
        'savefig.dpi': 150,
    })
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)
print('Setup complete.')

---

## 2. Scaled Dot-Product Attention

We implement the core attention function:

$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
# === 2.1 Implement Scaled Dot-Product Attention ===

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask  # mask has -inf for masked positions
    weights = softmax(scores)
    output = weights @ V
    return output, weights

# Test with small example
n, d_k, d_v = 4, 8, 8
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)

print('Q shape:', Q.shape)
print('K shape:', K.shape)
print('V shape:', V.shape)
print('Output shape:', output.shape)
print()
print('Attention weights (each row sums to 1):')
print(weights)
print()
row_sums = weights.sum(axis=1)
print(f'Row sums: {row_sums}')
ok = np.allclose(row_sums, 1.0)
print(f"{'PASS' if ok else 'FAIL'} - All rows sum to 1")

ok2 = np.all(weights >= 0)
print(f"{'PASS' if ok2 else 'FAIL'} - All weights non-negative")

In [ ]:
# === 2.2 Visualise Attention Weights ===

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(weights, cmap='viridis', aspect='auto')
    fig.colorbar(im, ax=ax, label='Attention weight')
    ax.set_title('Attention weight matrix (random Q, K)')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center',
                    fontsize=10, color='white' if weights[i,j] < 0.5 else 'black')
    fig.tight_layout()
    plt.show()
    print('Attention heatmap displayed.')
else:
    print('Matplotlib not available, skipping plot.')

---

## 3. The Variance Problem and Scaling

**Theorem:** For $\mathbf{q}, \mathbf{k} \in \mathbb{R}^{d_k}$ with i.i.d. $\mathcal{N}(0,1)$ components: $\operatorname{Var}(\mathbf{q}^\top \mathbf{k}) = d_k$.

We verify this empirically and show the effect on softmax.

In [ ]:
# === 3.1 Empirical Verification of Variance Scaling ===

dims = [4, 8, 16, 32, 64, 128, 256, 512]
n_samples = 50000
empirical_vars_unscaled = []
empirical_vars_scaled = []

for d_k in dims:
    q = np.random.randn(n_samples, d_k)
    k = np.random.randn(n_samples, d_k)
    dots = np.sum(q * k, axis=1)  # q^T k for each sample
    empirical_vars_unscaled.append(np.var(dots))
    empirical_vars_scaled.append(np.var(dots / np.sqrt(d_k)))

print('d_k  | Var(q^T k) | Theoretical | Var(q^T k / sqrt(d_k))')
print('-' * 60)
for d, vu, vs in zip(dims, empirical_vars_unscaled, empirical_vars_scaled):
    print(f'{d:4d} | {vu:10.2f} | {d:11d} | {vs:.4f}')

# Verify
ok = all(abs(vu - d) / d < 0.1 for vu, d in zip(empirical_vars_unscaled, dims))
print(f"\n{'PASS' if ok else 'FAIL'} - Var(q^T k) ~ d_k for all dimensions")

ok2 = all(abs(vs - 1.0) < 0.15 for vs in empirical_vars_scaled)
print(f"{'PASS' if ok2 else 'FAIL'} - Var(q^T k / sqrt(d_k)) ~ 1 for all dimensions")

In [ ]:
# === 3.2 Visualise Variance Scaling ===

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(dims, empirical_vars_unscaled, 'o-', color=COLORS['primary'],
             label='Empirical Var($\mathbf{q}^\top\mathbf{k}$)', markersize=8)
    ax1.plot(dims, dims, '--', color=COLORS['error'],
             label='Theoretical: $d_k$', linewidth=2)
    ax1.set_xlabel('Dimension $d_k$')
    ax1.set_ylabel('Variance')
    ax1.set_title('Unscaled dot products: Var grows with $d_k$')
    ax1.legend()

    ax2.plot(dims, empirical_vars_scaled, 'o-', color=COLORS['tertiary'],
             label='Empirical Var($\mathbf{q}^\top\mathbf{k}/\sqrt{d_k}$)', markersize=8)
    ax2.axhline(1.0, color=COLORS['error'], linestyle='--',
                label='Theoretical: 1', linewidth=2)
    ax2.set_xlabel('Dimension $d_k$')
    ax2.set_ylabel('Variance')
    ax2.set_title('Scaled dot products: Var $\\approx 1$ regardless of $d_k$')
    ax2.legend()
    ax2.set_ylim(0, 2)

    fig.tight_layout()
    plt.show()
    print('Variance scaling plots displayed.')

In [ ]:
# === 3.3 Effect of Scaling on Softmax Entropy ===

def entropy(p):
    """Shannon entropy of a distribution."""
    p = p[p > 0]
    return -np.sum(p * np.log(p))

d_k = 64
n_keys = 20
n_trials = 1000

entropies_unscaled = []
entropies_scaled = []

for _ in range(n_trials):
    q = np.random.randn(d_k)
    K = np.random.randn(n_keys, d_k)
    scores_unscaled = K @ q
    scores_scaled = scores_unscaled / np.sqrt(d_k)
    entropies_unscaled.append(entropy(softmax(scores_unscaled)))
    entropies_scaled.append(entropy(softmax(scores_scaled)))

max_entropy = np.log(n_keys)
print(f'Max possible entropy (uniform over {n_keys} keys): {max_entropy:.4f}')
print(f'Mean entropy (unscaled, d_k={d_k}): {np.mean(entropies_unscaled):.4f}')
print(f'Mean entropy (scaled):              {np.mean(entropies_scaled):.4f}')
print()
print('Unscaled attention is much sharper (lower entropy) = softmax saturated')
print('Scaled attention preserves a healthy entropy range')

---

## 4. Softmax Properties

We explore the softmax function: temperature effects, Jacobian structure, and numerical stability.

In [ ]:
# === 4.1 Softmax Temperature ===

logits = np.array([2.0, 1.0, 0.5, -0.5, -1.0])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

print('Softmax output at different temperatures:')
print(f'{"T":>6s} | Distribution')
print('-' * 60)
for T in temperatures:
    probs = softmax(logits / T)
    dist_str = ' '.join(f'{p:.4f}' for p in probs)
    print(f'{T:6.1f} | {dist_str}  H={entropy(probs):.3f}')

print()
print('T -> 0: approaches one-hot (hard attention)')
print('T -> inf: approaches uniform (no attention)')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(logits))
    width = 0.12
    for i, T in enumerate(temperatures):
        probs = softmax(logits / T)
        ax.bar(x + i*width, probs, width, label=f'T={T}', alpha=0.85)
    ax.set_xlabel('Position')
    ax.set_ylabel('Attention weight')
    ax.set_title('Softmax temperature: from hard to uniform attention')
    ax.legend()
    ax.set_xticks(x + width * len(temperatures)/2)
    ax.set_xticklabels([f'pos {i}' for i in range(len(logits))])
    fig.tight_layout()
    plt.show()

In [ ]:
# === 4.2 Softmax Jacobian ===

def softmax_jacobian(s):
    """Compute Jacobian of softmax: diag(s) - s @ s.T"""
    return np.diag(s) - np.outer(s, s)

# Example
logits = np.array([2.0, 1.0, 0.5, -0.5, -1.0])
s = softmax(logits)
J = softmax_jacobian(s)

print('Softmax output:', s)
print()
print('Jacobian matrix:')
print(J)
print()

# Verify rank is n-1
rank = np.linalg.matrix_rank(J, tol=1e-10)
print(f'Rank of Jacobian: {rank} (expected {len(logits)-1})')
ok = rank == len(logits) - 1
print(f"{'PASS' if ok else 'FAIL'} - Jacobian has rank n-1")

# Verify J @ 1 = 0 (constraint sum = 1 removes one DOF)
ones = np.ones(len(logits))
ok2 = np.allclose(J @ ones, 0)
print(f"{'PASS' if ok2 else 'FAIL'} - J @ 1 = 0 (sum constraint)")

# Numerical Jacobian check
eps = 1e-5
J_numerical = np.zeros_like(J)
for i in range(len(logits)):
    e = np.zeros(len(logits))
    e[i] = eps
    J_numerical[:, i] = (softmax(logits + e) - softmax(logits - e)) / (2 * eps)
ok3 = np.allclose(J, J_numerical, atol=1e-6)
print(f"{'PASS' if ok3 else 'FAIL'} - Analytical Jacobian matches numerical")

---

## 5. Attention as Kernel Smoothing

Attention implements a Nadaraya-Watson kernel estimator with a softmax kernel.

In [ ]:
# === 5.1 Attention as Nadaraya-Watson Estimator ===

# 1D kernel regression example
np.random.seed(42)
n_data = 50
x_data = np.sort(np.random.uniform(-3, 3, n_data))
y_data = np.sin(x_data) + 0.2 * np.random.randn(n_data)

x_query = np.linspace(-3, 3, 200)

def nw_estimate(x_q, x_data, y_data, bandwidth):
    """Nadaraya-Watson kernel regression."""
    # Using softmax kernel: exp(q * k / bandwidth)
    scores = np.outer(x_q, x_data) / bandwidth
    weights = softmax(scores, axis=1)
    return weights @ y_data

bandwidths = [0.1, 0.5, 1.0, 3.0]

if HAS_MPL:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, bw in zip(axes, bandwidths):
        y_pred = nw_estimate(x_query, x_data, y_data, bw)
        ax.scatter(x_data, y_data, s=20, alpha=0.5, color=COLORS['neutral'],
                   label='Data')
        ax.plot(x_query, np.sin(x_query), '--', color=COLORS['tertiary'],
                label='True $\sin(x)$')
        ax.plot(x_query, y_pred, color=COLORS['primary'],
                label='NW estimate')
        ax.set_title(f'Bandwidth = {bw}')
        ax.set_xlabel('$x$')
        ax.set_ylabel('$y$')
        ax.legend(fontsize=9)
    fig.suptitle('Attention = Kernel Smoother (Nadaraya-Watson)', fontsize=14)
    fig.tight_layout()
    plt.show()
    print('Small bandwidth = sharp attention, large bandwidth = diffuse attention')
else:
    y_pred = nw_estimate(x_query, x_data, y_data, 1.0)
    mse = np.mean((y_pred - np.sin(x_query))**2)
    print(f'NW estimate MSE (bw=1.0): {mse:.4f}')

---

## 6. Multi-Head Attention

We implement multi-head attention from scratch: project Q, K, V into $h$ subspaces, run attention in parallel, concatenate, and project back.

In [ ]:
# === 6.1 Multi-Head Attention Implementation ===

class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        assert d_model % n_heads == 0

        # Random init (Xavier scale)
        scale = 1.0 / np.sqrt(d_model)
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale

    def forward(self, X, mask=None):
        n_seq = X.shape[0]
        Q = X @ self.W_Q  # (n, d_model)
        K = X @ self.W_K
        V = X @ self.W_V

        # Reshape into heads: (n, d_model) -> (n_heads, n, d_k)
        Q_heads = Q.reshape(n_seq, self.n_heads, self.d_k).transpose(1, 0, 2)
        K_heads = K.reshape(n_seq, self.n_heads, self.d_k).transpose(1, 0, 2)
        V_heads = V.reshape(n_seq, self.n_heads, self.d_k).transpose(1, 0, 2)

        # Attention per head
        all_heads = []
        all_weights = []
        for h in range(self.n_heads):
            out_h, w_h = scaled_dot_product_attention(
                Q_heads[h], K_heads[h], V_heads[h], mask)
            all_heads.append(out_h)
            all_weights.append(w_h)

        # Concatenate: (n_heads, n, d_k) -> (n, d_model)
        concat = np.concatenate(all_heads, axis=-1)
        output = concat @ self.W_O
        return output, all_weights

# Test
d_model, n_heads, n_seq = 64, 8, 10
X = np.random.randn(n_seq, d_model)
mha = MultiHeadAttention(d_model, n_heads)
output, attn_weights = mha.forward(X)

print(f'Input shape: {X.shape}')
print(f'Output shape: {output.shape}')
print(f'Number of heads: {len(attn_weights)}')
print(f'Attention weight shape per head: {attn_weights[0].shape}')

ok = output.shape == X.shape
print(f"\n{'PASS' if ok else 'FAIL'} - Output shape matches input shape")

# Parameter count
params = 4 * d_model * d_model
print(f'\nTotal MHA parameters: {params:,} = 4 * {d_model}^2')
print(f'This is independent of the number of heads ({n_heads})')

In [ ]:
# === 6.2 Visualise Per-Head Attention Patterns ===

if HAS_MPL:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for h, ax in enumerate(axes.flat):
        im = ax.imshow(attn_weights[h], cmap='viridis', aspect='auto')
        ax.set_title(f'Head {h+1}')
        if h >= 4:
            ax.set_xlabel('Key')
        if h % 4 == 0:
            ax.set_ylabel('Query')
    fig.suptitle(f'Attention patterns: {n_heads} heads, d_model={d_model}', fontsize=14)
    fig.tight_layout()
    plt.show()
    print('Each head learns different attention patterns.')
    print('Some heads may be sharp (low entropy), others diffuse (high entropy).')
else:
    for h in range(n_heads):
        ent = -np.sum(attn_weights[h] * np.log(attn_weights[h] + 1e-12), axis=1).mean()
        print(f'Head {h+1}: mean entropy = {ent:.3f}')

---

## 7. Positional Encodings

Without positional information, attention is permutation-equivariant. We verify this and implement sinusoidal encodings.

In [ ]:
# === 7.1 Verify Permutation Equivariance ===

n_seq, d_model = 6, 32
X = np.random.randn(n_seq, d_model)

# Random permutation
perm = np.random.permutation(n_seq)
X_perm = X[perm]

# Attention without positional encoding
mha_small = MultiHeadAttention(d_model, 4)
out_original, _ = mha_small.forward(X)
out_permuted, _ = mha_small.forward(X_perm)

# Check: permuting input and then attending == attending then permuting
ok = np.allclose(out_permuted, out_original[perm], atol=1e-10)
print(f"{'PASS' if ok else 'FAIL'} - Attention is permutation-equivariant")
print(f'  ||out_permuted - out_original[perm]|| = {la.norm(out_permuted - out_original[perm]):.2e}')
print()
print('This proves position must be explicitly injected.')

In [ ]:
# === 7.2 Sinusoidal Positional Encoding ===

def sinusoidal_encoding(max_len, d_model):
    """Compute sinusoidal positional encoding."""
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, np.newaxis]
    div_term = 10000 ** (2 * np.arange(d_model // 2) / d_model)
    PE[:, 0::2] = np.sin(pos / div_term)
    PE[:, 1::2] = np.cos(pos / div_term)
    return PE

PE = sinusoidal_encoding(128, 64)
print(f'PE shape: {PE.shape}')
print(f'PE[0, :8] = {PE[0, :8]}')
print(f'PE[1, :8] = {PE[1, :8]}')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    im = ax1.imshow(PE, cmap='RdBu_r', aspect='auto')
    fig.colorbar(im, ax=ax1, label='Encoding value')
    ax1.set_title('Sinusoidal Positional Encoding')
    ax1.set_xlabel('Dimension')
    ax1.set_ylabel('Position')

    # Dot product between positions = similarity
    sim = PE @ PE.T
    im2 = ax2.imshow(sim[:32, :32], cmap='viridis', aspect='auto')
    fig.colorbar(im2, ax=ax2, label='Dot product')
    ax2.set_title('Position similarity (PE dot products)')
    ax2.set_xlabel('Position $j$')
    ax2.set_ylabel('Position $i$')

    fig.tight_layout()
    plt.show()
    print('Low dimensions = low frequency, high dimensions = high frequency')
    print('Nearby positions have higher similarity (brighter diagonal)')

---

## 8. Rotary Position Embedding (RoPE)

RoPE encodes position by rotating query/key vectors. The attention score depends only on relative position $m - n$.

In [ ]:
# === 8.1 RoPE Implementation ===

def rope_rotation(x, pos, d_model, base=10000):
    """Apply RoPE rotation to a vector at given position."""
    x_rot = x.copy()
    for i in range(d_model // 2):
        theta = pos / (base ** (2 * i / d_model))
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        x0 = x[2*i]
        x1 = x[2*i + 1]
        x_rot[2*i]     = x0 * cos_t - x1 * sin_t
        x_rot[2*i + 1] = x0 * sin_t + x1 * cos_t
    return x_rot

# Verify: score depends only on relative position
d = 16
q = np.random.randn(d)
k = np.random.randn(d)

# Test multiple (m, n) pairs with same relative position delta = 4
delta = 4
scores = []
for m in [0, 3, 7, 15, 42]:
    n = m + delta
    q_rot = rope_rotation(q, m, d)
    k_rot = rope_rotation(k, n, d)
    score = q_rot @ k_rot
    scores.append(score)
    print(f'm={m:2d}, n={n:2d}, delta={delta}: score = {score:.8f}')

ok = np.allclose(scores, scores[0], atol=1e-10)
print(f"\n{'PASS' if ok else 'FAIL'} - Score depends only on relative position (delta={delta})")

# Also verify different deltas give different scores
scores_d2 = []
for m in [0, 5, 10]:
    q_rot = rope_rotation(q, m, d)
    k_rot = rope_rotation(k, m + 2, d)
    scores_d2.append(q_rot @ k_rot)

ok2 = np.allclose(scores_d2, scores_d2[0], atol=1e-10)
print(f"{'PASS' if ok2 else 'FAIL'} - Score for delta=2 is also consistent")
print(f'Score(delta=4) = {scores[0]:.6f}, Score(delta=2) = {scores_d2[0]:.6f}')

In [ ]:
# === 8.2 RoPE Frequency Spectrum ===

d_model = 64
base = 10000
freqs = [1.0 / (base ** (2 * i / d_model)) for i in range(d_model // 2)]
wavelengths = [2 * np.pi / f for f in freqs]

print(f'Number of frequency pairs: {d_model // 2}')
print(f'Shortest wavelength: {min(wavelengths):.1f} positions')
print(f'Longest wavelength: {max(wavelengths):.1f} positions')
print(f'Frequency range: {min(freqs):.2e} to {max(freqs):.2e}')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.semilogy(range(d_model // 2), freqs, 'o-', color=COLORS['primary'],
                 markersize=4)
    ax1.set_xlabel('Dimension pair index $i$')
    ax1.set_ylabel('Frequency $\\theta_i$')
    ax1.set_title('RoPE frequency spectrum (log scale)')

    # Show rotation at different positions for dim pair 0 vs dim pair 15
    positions = np.arange(100)
    for idx, label in [(0, 'Pair 0 (high freq)'), (15, 'Pair 15 (mid freq)'),
                        (31, 'Pair 31 (low freq)')]:
        angles = positions * freqs[idx]
        ax2.plot(positions, np.cos(angles), label=label)
    ax2.set_xlabel('Position')
    ax2.set_ylabel('cos(pos $\\cdot$ freq)')
    ax2.set_title('RoPE rotation angle vs position')
    ax2.legend()

    fig.tight_layout()
    plt.show()

---

## 9. Feed-Forward Networks

We implement the position-wise FFN and explore the FFN-as-memory interpretation.

In [ ]:
# === 9.1 FFN: ReLU vs SwiGLU ===

def ffn_relu(x, W1, b1, W2, b2):
    """Standard FFN with ReLU."""
    h = np.maximum(0, x @ W1.T + b1)  # ReLU
    return h @ W2.T + b2

def swish(x):
    return x / (1 + np.exp(-x))

def ffn_swiglu(x, W1, W3, W2):
    """SwiGLU FFN (no bias, as in LLaMA)."""
    gate = swish(x @ W1.T)
    up = x @ W3.T
    return (gate * up) @ W2.T

d_model = 64
d_ff_relu = 4 * d_model  # 256
d_ff_swiglu = int(8/3 * d_model)  # ~170

# ReLU FFN
W1_r = np.random.randn(d_ff_relu, d_model) * 0.02
b1_r = np.zeros(d_ff_relu)
W2_r = np.random.randn(d_model, d_ff_relu) * 0.02
b2_r = np.zeros(d_model)

# SwiGLU FFN
W1_s = np.random.randn(d_ff_swiglu, d_model) * 0.02
W3_s = np.random.randn(d_ff_swiglu, d_model) * 0.02
W2_s = np.random.randn(d_model, d_ff_swiglu) * 0.02

x = np.random.randn(d_model)
out_relu = ffn_relu(x, W1_r, b1_r, W2_r, b2_r)
out_swiglu = ffn_swiglu(x, W1_s, W3_s, W2_s)

params_relu = 2 * d_model * d_ff_relu + d_ff_relu + d_model
params_swiglu = 3 * d_model * d_ff_swiglu

print(f'ReLU FFN: d_ff={d_ff_relu}, params={params_relu:,}')
print(f'SwiGLU FFN: d_ff={d_ff_swiglu}, params={params_swiglu:,}')
print(f'Param ratio: {params_swiglu / params_relu:.2f}')
print(f'Output shape: {out_relu.shape}')

ok = out_relu.shape == (d_model,)
print(f"\n{'PASS' if ok else 'FAIL'} - FFN output has correct shape")

In [ ]:
# === 9.2 FFN as Key-Value Memory ===

# Decompose FFN(x) = sum_i relu(k_i^T x) * v_i
# where k_i = row i of W1, v_i = column i of W2

x = np.random.randn(d_model) * 0.5

# Method 1: Standard FFN
h = np.maximum(0, W1_r @ x + b1_r)
out_standard = W2_r.T @ h + b2_r  # note: W2 is (d_model, d_ff), need transpose

# Method 2: Sum over key-value pairs
out_memory = np.zeros(d_model)
active_neurons = 0
for i in range(d_ff_relu):
    activation = max(0, W1_r[i] @ x + b1_r[i])
    if activation > 0:
        out_memory += activation * W2_r[:, i]
        active_neurons += 1
out_memory += b2_r

ok = np.allclose(out_standard, out_memory)
print(f"{'PASS' if ok else 'FAIL'} - FFN equals sum of key-value memory lookups")
print(f'Active neurons (ReLU > 0): {active_neurons} / {d_ff_relu} = {active_neurons/d_ff_relu:.1%}')
print('ReLU sparsity means only a fraction of memory slots are active per input.')

---

## 10. Normalisation Layers

We implement LayerNorm and RMSNorm, verify their properties, and compare.

In [ ]:
# === 10.1 LayerNorm vs RMSNorm ===

def layer_norm(x, gamma, beta, eps=1e-5):
    mu = np.mean(x)
    sigma2 = np.var(x)
    x_hat = (x - mu) / np.sqrt(sigma2 + eps)
    return gamma * x_hat + beta

def rms_norm(x, gamma, eps=1e-5):
    rms = np.sqrt(np.mean(x**2) + eps)
    return gamma * x / rms

d = 64
x = np.random.randn(d) * 3 + 2  # non-zero mean, large variance
gamma = np.ones(d)
beta = np.zeros(d)

ln_out = layer_norm(x, gamma, beta)
rms_out = rms_norm(x, gamma)

print(f'Input:     mean={np.mean(x):.3f}, std={np.std(x):.3f}')
print(f'LayerNorm: mean={np.mean(ln_out):.6f}, std={np.std(ln_out):.6f}')
print(f'RMSNorm:   mean={np.mean(rms_out):.3f}, RMS={np.sqrt(np.mean(rms_out**2)):.6f}')

# Verify LayerNorm output has zero mean, unit variance
ok = abs(np.mean(ln_out)) < 1e-10 and abs(np.std(ln_out) - 1.0) < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} - LayerNorm output has mean~0, std~1")

# Verify RMSNorm output has unit RMS
rms_val = np.sqrt(np.mean(rms_out**2))
ok2 = abs(rms_val - 1.0) < 0.01
print(f"{'PASS' if ok2 else 'FAIL'} - RMSNorm output has RMS~1")

# Parameter comparison
print(f'\nLayerNorm params: {2*d} (gamma + beta)')
print(f'RMSNorm params:   {d} (gamma only)')

---

## 11. Full Transformer Block

We assemble a complete Pre-Norm Transformer block: RMSNorm -> MHA -> residual -> RMSNorm -> FFN -> residual.

In [ ]:
# === 11.1 Complete Transformer Block ===

class TransformerBlock:
    def __init__(self, d_model, n_heads, d_ff):
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.gamma1 = np.ones(d_model)
        self.gamma2 = np.ones(d_model)
        scale = 0.02
        self.W1 = np.random.randn(d_ff, d_model) * scale
        self.W3 = np.random.randn(d_ff, d_model) * scale
        self.W2 = np.random.randn(d_model, d_ff) * scale

    def forward(self, X, mask=None):
        # Pre-norm + MHA + residual
        X_norm1 = np.array([rms_norm(X[i], self.gamma1) for i in range(len(X))])
        attn_out, weights = self.mha.forward(X_norm1, mask)
        Z = X + attn_out

        # Pre-norm + SwiGLU FFN + residual
        Z_norm = np.array([rms_norm(Z[i], self.gamma2) for i in range(len(Z))])
        ffn_out = np.array([ffn_swiglu(Z_norm[i], self.W1, self.W3, self.W2)
                            for i in range(len(Z_norm))])
        out = Z + ffn_out
        return out, weights

d_model, n_heads, d_ff = 64, 8, int(8/3 * 64)
n_seq = 10
X = np.random.randn(n_seq, d_model)

block = TransformerBlock(d_model, n_heads, d_ff)
out, _ = block.forward(X)

print(f'Input shape:  {X.shape}')
print(f'Output shape: {out.shape}')
ok = out.shape == X.shape
print(f"{'PASS' if ok else 'FAIL'} - Block preserves shape (essential for residual)")

# Check residual contribution
residual_norm = la.norm(out - X)
input_norm = la.norm(X)
print(f'\n||residual|| / ||input|| = {residual_norm/input_norm:.4f}')
print('Small ratio means the block adds a small perturbation (as intended at init).')

In [ ]:
# === 11.2 Causal Masking ===

def create_causal_mask(n):
    """Create causal attention mask: -inf for future positions."""
    mask = np.full((n, n), -1e9)
    mask = np.triu(mask, k=1)  # upper triangle = -inf
    return mask

n = 6
mask = create_causal_mask(n)
print('Causal mask (0 = attend, -1e9 = blocked):')
print((mask == 0).astype(int))

# Compare masked vs unmasked attention
Q = np.random.randn(n, 16)
K = np.random.randn(n, 16)
V = np.random.randn(n, 16)

out_full, w_full = scaled_dot_product_attention(Q, K, V)
out_causal, w_causal = scaled_dot_product_attention(Q, K, V, mask)

# Verify causal mask is lower-triangular
is_lower_tri = np.allclose(np.triu(w_causal, k=1), 0, atol=1e-6)
print(f"\n{'PASS' if is_lower_tri else 'FAIL'} - Causal attention is lower-triangular")

# Verify position i output only depends on positions <= i
# Change position 5's key/value and check position 0's output
K_modified = K.copy()
K_modified[5] += 100  # big change at position 5
out_modified, _ = scaled_dot_product_attention(Q, K_modified, V, mask)

# Position 0 should be unaffected (can't see position 5)
ok = np.allclose(out_causal[0], out_modified[0])
print(f"{'PASS' if ok else 'FAIL'} - Position 0 unaffected by changes at position 5")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(w_full, cmap='viridis', aspect='auto')
    ax1.set_title('Full attention (bidirectional)')
    ax1.set_xlabel('Key'); ax1.set_ylabel('Query')
    ax2.imshow(w_causal, cmap='viridis', aspect='auto')
    ax2.set_title('Causal attention (lower triangular)')
    ax2.set_xlabel('Key'); ax2.set_ylabel('Query')
    fig.tight_layout()
    plt.show()

---

## 12. Parameter Counting

We verify the parameter counting formulas for real models.

In [ ]:
# === 12.1 Parameter Counting for Major Models ===

def count_params(L, d, h, d_ff, V, swiglu=False, tied_embeddings=True):
    """Count parameters in a decoder-only Transformer."""
    # Token embedding
    embed = V * d
    # Per-layer attention: W_Q, W_K, W_V, W_O
    attn_per_layer = 4 * d * d
    # Per-layer FFN
    if swiglu:
        ffn_per_layer = 3 * d * d_ff  # W1, W3, W2 (no bias)
    else:
        ffn_per_layer = 2 * d * d_ff + d_ff + d  # W1, W2 + biases
    # Per-layer norms (2x RMSNorm)
    norm_per_layer = 2 * d
    # Final norm
    final_norm = d
    # LM head
    lm_head = 0 if tied_embeddings else V * d

    total = embed + L * (attn_per_layer + ffn_per_layer + norm_per_layer) + final_norm + lm_head
    return total, {
        'embedding': embed,
        'attn_per_layer': attn_per_layer,
        'ffn_per_layer': ffn_per_layer,
        'total_per_layer': attn_per_layer + ffn_per_layer + norm_per_layer
    }

models = [
    ('GPT-2 Small',  12,   768,  12,  3072,  50257, False, True),
    ('LLaMA-7B',     32,  4096,  32, 11008,  32000, True,  False),
    ('LLaMA-13B',    40,  5120,  40, 13824,  32000, True,  False),
    ('LLaMA-70B',    80,  8192,  64, 28672,  32000, True,  False),
    ('Mistral-7B',   32,  4096,  32, 14336,  32000, True,  False),
]

print(f'{"Model":<15s} {"Computed":>12s} {"Published":>12s} {"Match?":>8s}')
print('-' * 55)
published = [124e6, 6.7e9, 13e9, 70e9, 7.2e9]
for (name, L, d, h, d_ff, V, swiglu, tied), pub in zip(models, published):
    total, breakdown = count_params(L, d, h, d_ff, V, swiglu, tied)
    match = 'OK' if abs(total - pub) / pub < 0.15 else 'DIFF'
    print(f'{name:<15s} {total/1e9:12.2f}B {pub/1e9:12.2f}B {match:>8s}')

# Detailed breakdown for LLaMA-7B
total, bd = count_params(32, 4096, 32, 11008, 32000, True, False)
print(f'\nLLaMA-7B breakdown:')
print(f'  Embedding:    {32000*4096/1e9:.2f}B')
print(f'  Attn/layer:   {bd["attn_per_layer"]/1e6:.1f}M')
print(f'  FFN/layer:    {bd["ffn_per_layer"]/1e6:.1f}M')
print(f'  Total/layer:  {bd["total_per_layer"]/1e6:.1f}M')
print(f'  32 layers:    {32*bd["total_per_layer"]/1e9:.2f}B')
print(f'  TOTAL:        {total/1e9:.2f}B')

---

## 13. Computational Complexity

We compute FLOPs for attention and FFN, and show when each dominates.

In [ ]:
# === 13.1 FLOPs Analysis ===

def compute_flops(n, d, d_ff, L):
    """Compute FLOPs for one forward pass."""
    # Attention: QKV projections + score + AV + output projection
    attn_flops = 8 * n * d**2 + 4 * n**2 * d
    # FFN: SwiGLU has 3 matmuls
    ffn_flops = 3 * 2 * n * d * d_ff  # 6 * n * d * d_ff
    total_per_layer = attn_flops + ffn_flops
    return L * total_per_layer, L * attn_flops, L * ffn_flops

d = 4096
d_ff = 11008
L = 32
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]

print(f'd_model={d}, d_ff={d_ff}, L={L}')
print(f'{"n":>8s} {"Attn TFLOPs":>14s} {"FFN TFLOPs":>14s} {"Total TFLOPs":>14s} {"Attn%":>7s}')
print('-' * 65)

attn_fracs = []
for n in seq_lengths:
    total, attn, ffn = compute_flops(n, d, d_ff, L)
    attn_frac = attn / total * 100
    attn_fracs.append(attn_frac)
    print(f'{n:8d} {attn/1e12:14.1f} {ffn/1e12:14.1f} {total/1e12:14.1f} {attn_frac:6.1f}%')

# Crossover point
crossover = 6 * d  # when 4n^2d = 24nd^2 -> n = 6d
print(f'\nCrossover: n = 6d = {crossover:,} (attn cost = FFN cost)')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(seq_lengths, attn_fracs, 'o-', color=COLORS['primary'], label='Attention %')
    ax.axhline(50, color=COLORS['neutral'], linestyle='--', label='50% crossover')
    ax.axvline(crossover, color=COLORS['error'], linestyle=':', label=f'n = 6d = {crossover:,}')
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Sequence length $n$')
    ax.set_ylabel('Attention fraction of total FLOPs (%)')
    ax.set_title('When does attention dominate compute?')
    ax.legend()
    fig.tight_layout()
    plt.show()

---

## 14. FlashAttention: Online Softmax

We implement the online softmax trick that enables block-wise attention computation.

In [ ]:
# === 14.1 Online Softmax Trick ===

def standard_attention(Q, K, V):
    """Standard O(n^2) memory attention."""
    d_k = Q.shape[-1]
    S = Q @ K.T / np.sqrt(d_k)
    A = softmax(S)
    return A @ V

def flash_attention(Q, K, V, block_size=2):
    """Block-wise attention with online softmax (FlashAttention simplified)."""
    n, d_k = Q.shape
    d_v = V.shape[1]
    O = np.zeros((n, d_v))  # output accumulator
    m = np.full(n, -np.inf)  # running max
    l = np.zeros(n)  # running sum-exp

    n_blocks = (n + block_size - 1) // block_size

    for j_block in range(n_blocks):
        j_start = j_block * block_size
        j_end = min(j_start + block_size, n)

        K_j = K[j_start:j_end]  # block of keys
        V_j = V[j_start:j_end]  # block of values

        # Compute scores for this block
        S_j = Q @ K_j.T / np.sqrt(d_k)  # (n, block_size)

        # Online softmax update
        m_new_block = np.max(S_j, axis=1)  # max in this block
        m_new = np.maximum(m, m_new_block)  # global running max

        # Rescale previous accumulator
        exp_diff_old = np.exp(m - m_new)
        exp_diff_new = np.exp(m_new_block - m_new)

        # New exponentials
        P_j = np.exp(S_j - m_new[:, None])  # (n, block_size)
        l_new_block = np.sum(P_j, axis=1)

        # Update
        l_new = exp_diff_old * l + l_new_block
        O = (exp_diff_old[:, None] * l[:, None] * O + P_j @ V_j) / l_new[:, None]

        m = m_new
        l = l_new

    return O

# Test
n, d = 8, 16
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

out_standard = standard_attention(Q, K, V)
out_flash = flash_attention(Q, K, V, block_size=2)

diff = la.norm(out_standard - out_flash)
ok = diff < 1e-10
print(f"{'PASS' if ok else 'FAIL'} - FlashAttention matches standard attention")
print(f'||difference|| = {diff:.2e}')
print(f'\nStandard: materialises {n}x{n} = {n*n} attention matrix')
print(f'Flash: processes in blocks of 2, peak memory O(n) not O(n^2)')

---

## 15. LoRA: Low-Rank Adaptation

We implement LoRA and verify the parameter savings.

In [ ]:
# === 15.1 LoRA Implementation ===

class LoRALayer:
    def __init__(self, W0, rank, alpha=None):
        """LoRA: W = W0 + (alpha/r) * B @ A"""
        self.W0 = W0  # frozen pretrained weight
        self.rank = rank
        d_out, d_in = W0.shape
        self.alpha = alpha if alpha is not None else rank
        # A: (r, d_in), init from N(0, 1/d_in)
        self.A = np.random.randn(rank, d_in) / np.sqrt(d_in)
        # B: (d_out, r), init to zero
        self.B = np.zeros((d_out, rank))

    def forward(self, x):
        base = x @ self.W0.T
        lora = (self.alpha / self.rank) * (x @ self.A.T @ self.B.T)
        return base + lora

    @property
    def delta_W(self):
        return (self.alpha / self.rank) * self.B @ self.A

    @property
    def effective_W(self):
        return self.W0 + self.delta_W

# Create pretrained weight and LoRA adapter
d = 64
r = 4
W0 = np.random.randn(d, d) * 0.02  # pretrained
lora = LoRALayer(W0, rank=r)

# At init, delta_W = 0 (B is zero)
ok = np.allclose(lora.delta_W, 0)
print(f"{'PASS' if ok else 'FAIL'} - At init, delta_W = 0 (starts at pretrained)")

# Simulate training: update B
lora.B = np.random.randn(d, r) * 0.01

# Check rank of delta_W
U, S, Vt = la.svd(lora.delta_W)
rank_delta = np.sum(S > 1e-10)
print(f"\n{'PASS' if rank_delta == r else 'FAIL'} - rank(delta_W) = {rank_delta} (expected {r})")

# Parameter savings
full_params = d * d
lora_params = 2 * r * d
ratio = full_params / lora_params
print(f'\nFull fine-tuning: {full_params:,} params')
print(f'LoRA (r={r}):      {lora_params:,} params')
print(f'Reduction:         {ratio:.0f}x')

# Verify forward pass works
x = np.random.randn(d)
out_base = x @ W0.T
out_lora = lora.forward(x)
out_effective = x @ lora.effective_W.T
ok2 = np.allclose(out_lora, out_effective)
print(f"\n{'PASS' if ok2 else 'FAIL'} - LoRA forward == effective W forward")

---

## 16. Attention Patterns and Interpretability

We simulate different types of attention patterns found in trained Transformers.

In [ ]:
# === 16.1 Simulated Attention Patterns ===

n = 12  # sequence length

# Previous-token head: attend to position i-1
prev_token = np.zeros((n, n))
for i in range(1, n):
    prev_token[i, i-1] = 10.0
prev_token[0, 0] = 10.0
prev_token = softmax(prev_token)

# Diagonal head: attend to self
diagonal = np.eye(n) * 10
diagonal = softmax(diagonal)

# First-token head: always attend to position 0 (attention sink)
first_token = np.zeros((n, n))
first_token[:, 0] = 10.0
first_token = softmax(first_token)

# Broad/uniform head
broad = np.ones((n, n))
# Apply causal mask
broad = np.tril(broad)
broad = broad / broad.sum(axis=1, keepdims=True)

# Induction-like: attend to position where previous token matches
# Simulated: token 0=A, 1=B, 2=C, 3=A, 4=B -> position 4 attends to 1 (B follows A)
induction = np.eye(n) * 0.1
induction = np.tril(induction)
induction[4, 1] = 10  # pos 4 (after A at pos 3) looks at pos 1 (after A at pos 0)
induction[3, 0] = 5
induction = softmax(induction)

if HAS_MPL:
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    patterns = [
        (prev_token, 'Previous-token head'),
        (diagonal, 'Diagonal (self) head'),
        (first_token, 'First-token (sink) head'),
        (broad, 'Broad/uniform head'),
        (induction, 'Induction-like head'),
    ]
    for ax, (pattern, title) in zip(axes, patterns):
        im = ax.imshow(pattern, cmap='viridis', aspect='auto',
                       vmin=0, vmax=1)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Key')
        ax.set_ylabel('Query')
    fig.suptitle('Types of Attention Patterns in Trained Transformers', fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    for pattern, title in patterns:
        ent = -np.sum(pattern * np.log(pattern + 1e-12), axis=1).mean()
        print(f'{title}: mean entropy = {ent:.3f}')

In [ ]:
# === 16.2 Residual Stream Decomposition ===

# Simulate a 3-layer transformer and decompose the residual
d_model = 32
n_layers = 3
n_heads = 4

x0 = np.random.randn(d_model) * 0.1  # initial embedding

# Simulate layer contributions
np.random.seed(42)
attn_contributions = [np.random.randn(d_model) * 0.02 / np.sqrt(l+1)
                      for l in range(n_layers)]
ffn_contributions = [np.random.randn(d_model) * 0.03 / np.sqrt(l+1)
                     for l in range(n_layers)]

# Build residual stream
x = x0.copy()
for l in range(n_layers):
    x = x + attn_contributions[l] + ffn_contributions[l]

# Verify decomposition: x = x0 + sum of contributions
x_decomposed = x0 + sum(attn_contributions) + sum(ffn_contributions)
ok = np.allclose(x, x_decomposed)
print(f"{'PASS' if ok else 'FAIL'} - Residual stream = embedding + sum of layer contributions")

# Logit attribution: project each contribution onto a random 'unembedding' direction
u = np.random.randn(d_model)  # unembedding vector for some token
u = u / la.norm(u)

print('\nLogit contributions by component:')
print(f'  Embedding:    {x0 @ u:+.4f}')
for l in range(n_layers):
    print(f'  Layer {l} Attn:  {attn_contributions[l] @ u:+.4f}')
    print(f'  Layer {l} FFN:   {ffn_contributions[l] @ u:+.4f}')
print(f'  Total logit:  {x @ u:+.4f}')

# Verify sum
total = x0 @ u + sum(a @ u for a in attn_contributions) + sum(f @ u for f in ffn_contributions)
ok2 = np.allclose(total, x @ u)
print(f"\n{'PASS' if ok2 else 'FAIL'} - Logit = sum of per-component contributions")

---

## 9b. Activation Function Comparison

We compare ReLU, GELU, Swish, and SwiGLU activations.

In [ ]:
# === 9b.1 Activation Functions ===

x = np.linspace(-4, 4, 500)

relu = np.maximum(0, x)
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
swish_vals = x / (1 + np.exp(-x))
sigmoid = 1 / (1 + np.exp(-x))

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(x, relu, color=COLORS['primary'], label='ReLU')
    ax1.plot(x, gelu, color=COLORS['secondary'], label='GELU')
    ax1.plot(x, swish_vals, color=COLORS['tertiary'], label='Swish/SiLU')
    ax1.axhline(0, color=COLORS['neutral'], linewidth=0.5)
    ax1.axvline(0, color=COLORS['neutral'], linewidth=0.5)
    ax1.set_xlabel('$x$')
    ax1.set_ylabel('$\\sigma(x)$')
    ax1.set_title('Activation functions')
    ax1.legend()

    # Derivatives
    relu_grad = (x > 0).astype(float)
    swish_grad = sigmoid + x * sigmoid * (1 - sigmoid)
    gelu_grad = np.gradient(gelu, x)

    ax2.plot(x, relu_grad, color=COLORS['primary'], label='ReLU grad')
    ax2.plot(x, gelu_grad, color=COLORS['secondary'], label='GELU grad')
    ax2.plot(x, swish_grad, color=COLORS['tertiary'], label='Swish grad')
    ax2.axhline(0, color=COLORS['neutral'], linewidth=0.5)
    ax2.set_xlabel('$x$')
    ax2.set_ylabel('$\\sigma\'(x)$')
    ax2.set_title('Activation gradients')
    ax2.legend()

    fig.tight_layout()
    plt.show()
    print('ReLU: zero gradient for x<0 (dead neurons)')
    print('GELU/Swish: smooth, non-zero gradient for x<0')
else:
    print(f'ReLU(1)={max(0,1)}, GELU(1)~0.841, Swish(1)~0.731')

---

## 6b. Grouped-Query Attention (GQA)

GQA shares K/V heads across groups of Q heads, reducing KV cache size.

In [ ]:
# === 6b.1 GQA vs MHA KV Cache Comparison ===

def kv_cache_size(n_seq, n_kv_heads, d_head, n_layers, bytes_per_elem=2):
    """KV cache size in bytes."""
    return 2 * n_layers * n_seq * n_kv_heads * d_head * bytes_per_elem

configs = [
    ('MHA (64 KV heads)',  64, 128, 80),  # LLaMA-70B standard
    ('GQA (8 KV heads)',    8, 128, 80),  # LLaMA-2 70B actual
    ('MQA (1 KV head)',     1, 128, 80),  # Extreme sharing
]

n_seq = 4096
print(f'KV cache size for n_seq={n_seq}, d_head=128, L=80, fp16:')
print(f'{"Config":<22s} {"KV heads":>10s} {"Cache (GB)":>12s} {"Reduction":>12s}')
print('-' * 60)
base = None
for name, n_kv, d_head, L in configs:
    size = kv_cache_size(n_seq, n_kv, d_head, L)
    if base is None:
        base = size
    print(f'{name:<22s} {n_kv:>10d} {size/1e9:>12.2f} {base/size:>11.0f}x')

print(f'\nGQA reduces KV cache by {64/8:.0f}x while retaining ~99% of MHA quality.')
print('This is why LLaMA-2 70B uses GQA with 8 KV heads.')

---

## 13b. Scaling Laws Visualisation

The Chinchilla scaling law: $L(N, D) = A/N^\alpha + B/D^\beta + L_\infty$

In [ ]:
# === 13b.1 Chinchilla Scaling Law ===

# Approximate Chinchilla parameters
A = 406.4
B = 410.7
alpha = 0.34
beta = 0.28
L_inf = 1.69  # irreducible loss (nats)

def chinchilla_loss(N, D):
    return A / N**alpha + B / D**beta + L_inf

# Compute optimal N and D for given compute budget C = 6*N*D
compute_budgets = np.logspace(18, 25, 50)
optimal_N = []
optimal_D = []
optimal_L = []

for C in compute_budgets:
    # Grid search for optimal N given C = 6*N*D
    best_loss = float('inf')
    best_n = 0
    for log_n in np.linspace(6, 12, 200):
        N = 10**log_n
        D = C / (6 * N)
        if D < 1e6:
            continue
        loss = chinchilla_loss(N, D)
        if loss < best_loss:
            best_loss = loss
            best_n = N
    optimal_N.append(best_n)
    optimal_D.append(C / (6 * best_n))
    optimal_L.append(best_loss)

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.loglog(compute_budgets, optimal_N, color=COLORS['primary'],
              label='Optimal params $N$')
    ax1.loglog(compute_budgets, optimal_D, color=COLORS['secondary'],
              label='Optimal tokens $D$')
    ax1.set_xlabel('Compute budget $C$ (FLOPs)')
    ax1.set_ylabel('Count')
    ax1.set_title('Chinchilla: Optimal $N$ and $D$ vs compute')
    ax1.legend()

    ax2.semilogx(compute_budgets, optimal_L, color=COLORS['primary'])
    ax2.set_xlabel('Compute budget $C$ (FLOPs)')
    ax2.set_ylabel('Loss (nats)')
    ax2.set_title('Chinchilla: Optimal loss vs compute')
    ax2.axhline(L_inf, color=COLORS['error'], linestyle='--',
               label=f'$L_\\infty$ = {L_inf}')
    ax2.legend()

    fig.tight_layout()
    plt.show()

# Compute tokens/param ratio
ratios = [d/n for n, d in zip(optimal_N, optimal_D)]
print(f'Optimal tokens/param ratio: {np.mean(ratios):.1f}')
print('Chinchilla recommends ~20 tokens per parameter.')

In [ ]:
# === 13b.2 Memory Breakdown During Inference ===

# LLaMA-2 70B parameters
L = 80
d = 8192
n_kv_heads = 8
d_head = 128

seq_lengths = [1024, 2048, 4096, 8192, 16384, 32768]

print('LLaMA-2 70B KV Cache Growth (GQA with 8 KV heads, fp16):')
print(f'{"Seq len":>10s} {"KV cache":>12s} {"Model":>12s} {"KV/Model":>10s}')
print('-' * 50)

model_size_gb = 70e9 * 2 / 1e9  # 140 GB in fp16

for n in seq_lengths:
    kv_bytes = 2 * L * n * n_kv_heads * d_head * 2  # fp16
    kv_gb = kv_bytes / 1e9
    print(f'{n:>10d} {kv_gb:>11.1f}GB {model_size_gb:>11.1f}GB {kv_gb/model_size_gb:>9.1%}')

print(f'\nAt 32K context, KV cache is {2*80*32768*8*128*2/1e9:.1f}GB')
print('Without GQA (64 KV heads), it would be 8x larger!')

In [ ]:
# === 11b. Signal Propagation Through Layers ===

# Simulate residual stream growth through depth
d_model = 64
n_layers = 100

np.random.seed(42)
x = np.random.randn(d_model) * 0.1

norms_with_residual = [la.norm(x)]
norms_without_residual = [la.norm(x)]

x_res = x.copy()
x_nores = x.copy()

for l in range(n_layers):
    # Sublayer output (random, small)
    delta = np.random.randn(d_model) * 0.02 / np.sqrt(2 * (l + 1))

    # With residual
    x_res = x_res + delta
    norms_with_residual.append(la.norm(x_res))

    # Without residual (just the sublayer output)
    W = np.random.randn(d_model, d_model) * 0.95 / np.sqrt(d_model)
    x_nores = np.tanh(W @ x_nores)
    norms_without_residual.append(la.norm(x_nores))

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(norms_with_residual, color=COLORS['primary'],
            label='With residual connections')
    ax.plot(norms_without_residual, color=COLORS['error'],
            label='Without residual (vanilla)')
    ax.set_xlabel('Layer')
    ax.set_ylabel('||x|| (signal magnitude)')
    ax.set_title('Signal propagation: residual vs vanilla (100 layers)')
    ax.legend()
    ax.set_yscale('log')
    fig.tight_layout()
    plt.show()

print(f'After {n_layers} layers:')
print(f'  With residual:    ||x|| = {norms_with_residual[-1]:.4f}')
print(f'  Without residual: ||x|| = {norms_without_residual[-1]:.6f}')
print('Residual connections prevent signal from vanishing!')

---

## Summary

This notebook demonstrated:

1. **Scaled dot-product attention** — implementation, variance scaling proof, softmax temperature effects
2. **Multi-head attention** — parallel subspace projections, per-head patterns
3. **Positional encodings** — permutation equivariance proof, sinusoidal encoding, RoPE rotation verification
4. **Feed-forward networks** — ReLU vs SwiGLU, FFN as key-value memory decomposition
5. **Normalisation** — LayerNorm vs RMSNorm properties
6. **Full Transformer block** — Pre-Norm architecture, causal masking
7. **Parameter counting** — verified formulas against published model sizes
8. **Computational complexity** — FLOPs analysis, attention/FFN crossover
9. **FlashAttention** — online softmax trick with block-wise processing
10. **LoRA** — low-rank adaptation, parameter savings, rank verification
11. **Interpretability** — attention patterns, residual stream decomposition, logit attribution

Every result was verified numerically with PASS/FAIL checks.